## 01 · Exploratory Data Analysis
Goal: understand the data and justify the modelling choices (deduplication, class imbalance,
hand-crafted features, text cleaning).

In [4]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from spam_detection.config import PROJECT_ROOT, load_config, resolve_path
from spam_detection.data import load_raw
from spam_detection.features import TextStats, clean_text

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = load_raw(resolve_path(load_config()["paths"]["raw_data"]))
df.head()

ModuleNotFoundError: No module named 'pandas'

### Data quality

In [ ]:
print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum())
print("Exact duplicated rows:", df.duplicated().sum())
print("Duplicated texts:", df["text"].duplicated().sum())

df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print("Shape after deduplication:", df.shape)

### Class distribution

In [ ]:
counts = df["label"].value_counts()
print(pd.DataFrame({"count": counts, "percent": (counts / len(df) * 100).round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind="bar", rot=0, ax=ax, color=["tab:blue", "tab:red"])
ax.set(title="Class distribution (ham vs spam)", xlabel="", ylabel="Messages")
fig.savefig(FIG_DIR / "eda_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Message length

In [ ]:
df["msg_length"] = df["text"].str.len()
print(df.groupby("label")["msg_length"].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 4))
for label, group in df.groupby("label"):
    ax.hist(group["msg_length"], bins=40, alpha=0.6, density=True, label=label)
ax.set(title="Message length by class", xlabel="Characters", ylabel="Density")
ax.legend()
fig.savefig(FIG_DIR / "eda_message_length.png", dpi=150, bbox_inches="tight")
plt.show()

### Hand-crafted features by class
Average value of each feature in `TextStats`: this is the evidence for including them.


In [ ]:
stats = pd.DataFrame(
    TextStats(log_transform=False).fit_transform(df["text"]), columns=TextStats.FEATURE_NAMES
)
stats["label"] = df["label"].values
stats.groupby("label").mean().T.round(2)

### Most frequent words per class (after cleaning, without English stop words)

In [ ]:
def top_words(texts, n=15):
    counter = Counter(
        w for t in texts for w in clean_text(t).split() if w not in ENGLISH_STOP_WORDS
    )
    return counter.most_common(n)


for label in ("ham", "spam"):
    print(label.upper(), top_words(df.loc[df["label"] == label, "text"]), "\n")


### Cleaning example

In [ ]:
example = df.loc[df["label"] == "spam", "text"].iloc[2]
print("RAW    :", example)
print("CLEANED:", clean_text(example))